In [1]:
import scanpy as sc 
import numpy as np
from collections import Counter

In [2]:
def stratified_sample_by_sex_age(adata, celltype_key="final_cell_type", sex_key="sex", age_key="age_group", n_per_type=5000, random_state=0):
    '''Sample evenly n cells per cell type by sex and age group'''
    
    np.random.seed(random_state)
    
    obs = adata.obs.copy()
    sampled_indices = []

    for ct, df_ct in obs.groupby(celltype_key):
        # total cells needed for this cell type
        n_target = n_per_type
        
        # group within cell type by sex and age_group
        groups = df_ct.groupby([sex_key, age_key])
        n_groups = len(groups)
        
        # cells to take from each group evenly
        base_n = n_target // n_groups
        remainder = n_target % n_groups
        
        for i, (_, df_group) in enumerate(groups):
            n = base_n + (1 if i < remainder else 0)  # distribute remainder
            n = min(n, len(df_group))  # can't take more than available
            sampled_indices.extend(np.random.choice(df_group.index, size=n, replace=False))
    
    return adata[sampled_indices].copy()

#### Since ATAC has fewer nuclei, it is more likely to be the limitation in terms of which cell types to use 

- Cardiomyocyte
- Endothelial
- Fibroblast
- Myeloid
- Pericyte

In [3]:
%%time
adata = sc.read_h5ad("../../../RNA/aggregated_analysis/07_final_RNA_without_scvi.h5ad")
adata

CPU times: user 14.1 s, sys: 1min 54s, total: 2min 9s
Wall time: 2min 35s


AnnData object with n_obs × n_vars = 2305964 × 16115
    obs: 'age', 'donor_id', 'sex', 'region', 'cell_type', 'disease', 'consistent_cell_type', 'study', 'technology', 'cell_or_nuclei', 'barcode', 'sample_id', 'age_status', 'tech_plus_study', 'disease_binary', 'decade', 'age_group', '_scvi_batch', '_scvi_labels', 'leiden_scVI', 'scvi_cell_type', 'redo_leiden_0.5', 'UMAP1', 'UMAP2', 'v2_scvi_cell_type', 'final_cell_type'
    obsm: 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'
    layers: 'counts'

### Filter to non-diseased postnatal donors, sampling evenly by sex and age status

In [4]:
filt_data = adata[(adata.obs.disease_binary == "N") & (adata.obs.age_status == "postnatal")].copy()

In [5]:
Counter(filt_data.obs.age_status)

Counter({'postnatal': 1141372})

In [6]:
subsampled_adata = stratified_sample_by_sex_age(filt_data, n_per_type=3000)

/mnt/data1/william/tmp/ipykernel_4036620/2880677226.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for ct, df_ct in obs.groupby(celltype_key):
/mnt/data1/william/tmp/ipykernel_4036620/2880677226.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df_ct.groupby([sex_key, age_key])
/mnt/data1/william/tmp/ipykernel_4036620/2880677226.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df_ct.groupby(

In [7]:
Counter(subsampled_adata.obs['age_group'])

Counter({'old': 13000, 'middle': 12882, 'young': 11421})

#### confirm that function worked

In [8]:
CM_adata = subsampled_adata[subsampled_adata.obs.final_cell_type == "Cardiomyocyte"]

In [9]:
Counter(CM_adata.obs['age_group'])

Counter({'middle': 1000, 'old': 1000, 'young': 1000})

In [10]:
Counter(CM_adata.obs['sex'])

Counter({'female': 1500, 'male': 1500})

In [11]:
# add sex status and age status
subsampled_adata.obs['sex_and_age_status'] = subsampled_adata.obs['sex'].astype(str) + ":" + subsampled_adata.obs['age_group'].astype(str)

In [12]:
adata_metadata = subsampled_adata.obs 

In [13]:
adata_metadata.groupby(["sex_and_age_status", "final_cell_type"])['barcode'].count()

/mnt/data1/william/tmp/ipykernel_4036620/1815551823.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_metadata.groupby(["sex_and_age_status", "final_cell_type"])['barcode'].count()


sex_and_age_status  final_cell_type
female:middle       Adipocyte          500
                    Cardiomyocyte      500
                    Endocardial        500
                    Endothelial        500
                    Epicardial         382
                                      ... 
male:young          Mast               500
                    Myeloid            500
                    Neuronal           500
                    Pericyte           500
                    vSMC               500
Name: barcode, Length: 78, dtype: int64

In [14]:
### filter to the cell types for this analysis
cell_types = ["Cardiomyocyte", "Endothelial", "Fibroblast", "Myeloid", "Pericyte"]

In [15]:
adata_subsampled = subsampled_adata[subsampled_adata.obs.final_cell_type.isin(cell_types)].copy()
adata_subsampled.shape

(15000, 16115)

In [16]:
adata_subsampled.write("01_subsampled_RNA.h5ad")